In [45]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os
from arcpy.sa import ExtractByMask

# check out Spatial Analyst license
arcpy.CheckOutExtension("Spatial")

'CheckedOut'

In [46]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input bathy and slope rasters
gbc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\GBC15a_bathy_p26920_v6642"
gbc_slope = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\GBC15a_slope_p26920_v6642"
usc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_bathy_p26920_v6642"
usc_slope = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_slope_p26920_v6642"

# Output USC raster domain and outline polygon feature classes
rdomain_usc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\rdomain_USC10m_bathy_p26920_v6642"
rdomain_usc_slope = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\rdomain_USC10m_slope_p26920_v6642"
outline_usc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\outline_USC10m_bathy_p26920_v6642"
outline_usc_slope = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\outline_USC10m_slope_p26920_v6642"

# Output GBC rasters with OVERLAP REMOVED
NOOVRLP_gbc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\nooverlap_GBC15a_bathy_p26920_v6642"
NOOVRLP_gbc_slope = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\nooverlap_GBC15a_slope_p26920_v6642"

In [47]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun
arcpy.env.compression = "NONE"      # set compression to NONE for output rasters

# set environment coordinate system: horizontal EPSG:26920 + vertical EPSG:6642 (Extract by Mask tool does NOT carry vertical coordinate system of input(s) through to output layer(s), even if the vert coord system is present in all input layers - so must manually set coordinate system of output(s) to have both projected/geographic + vertical coordinate systems)
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(26920, 6642)

In [48]:
# ── 1. CREATE RASTER DOMAIN FEATURE CLASSES FOR USC RASTERS ──────────────────────────────────────────

# Raster Domain (3D Analyst Tools): Function
def create_raster_domains(in_raster, out_domain):
    try:
        arcpy.ddd.RasterDomain(
            in_raster=in_raster,
            out_feature_class=out_domain,
            out_geometry_type="POLYGON"
        )
        print(f"✅ Raster domain for {os.path.basename(in_raster)} raster successfully created: {out_domain} \n")

    # print warning if tool failed
    except Exception as e:
        print(f"❌[FAILURE] No raster domain created for {os.path.basename(out_domain)}: {e} \n")


# Call function for USC bathy + slope rasters
create_raster_domains(usc_bathy, rdomain_usc_bathy)
create_raster_domains(usc_slope, rdomain_usc_slope)

✅ Raster domain for USC10m_bathy_p26920_v6642 raster successfully created: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\rdomain_USC10m_bathy_p26920_v6642 

✅ Raster domain for USC10m_slope_p26920_v6642 raster successfully created: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\rdomain_USC10m_slope_p26920_v6642 



In [49]:
# ── 2. CREATE OUTLINE FEATURE CLASSES FOR USC RASTERS BY DISSOLVING LAND GAPS IN RASTER DOMAINS ──────

# Union (Analysis Tools) + Pairwise Dissolve (Analysis Tools): Function
def fill_rdomain_holes(in_domain, out_outline):
    try:

        # write Union result to temporary scratch feature class in memory
        union_domain = r"memory\union_domain"

        # Union (Analysis Tools):
        arcpy.analysis.Union(
            in_features=[in_domain],
            out_feature_class=union_domain,
            join_attributes="ALL",
            # "NO_GAPS" fills interior land holes by creating a polygon for each enclosed gap
            gaps="NO_GAPS"
        )

        # Pairwise Dissolve (Analysis Tools):
        arcpy.analysis.PairwiseDissolve(
            in_features=union_domain,
            out_feature_class=out_outline,
            # "SINGLE_PART" merges raster domain + all hole-fill polygons into single continuous-fill outline polygon
            multi_part="SINGLE_PART"
        )

        # clean up scratch Union result
        arcpy.management.Delete(union_domain)
        print(f"✅ Outline for {os.path.basename(in_domain)} successfully created: {out_outline} \n")

    # print warning if tool failed
    except Exception as e:
        print(f"❌[FAILURE] No outline created for {os.path.basename(out_outline)}: {e} \n")


# Call function for USC bathy + slope raster domains
fill_rdomain_holes(rdomain_usc_bathy, outline_usc_bathy)
fill_rdomain_holes(rdomain_usc_slope, outline_usc_slope)

✅ Outline for rdomain_USC10m_bathy_p26920_v6642 successfully created: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\outline_USC10m_bathy_p26920_v6642 

✅ Outline for rdomain_USC10m_slope_p26920_v6642 successfully created: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\outline_USC10m_slope_p26920_v6642 



In [50]:
# ── INSPECT INPUT GBC RASTERS BEFORE REMOVING OVERLAP ────────────────────────────────────────────────

# Function to print all info (number of bands, pixel type, min, max, cell size, coordinate systems) for each raster
def print_raster_info(raster):

    # get number of bands info
    d = arcpy.Describe(raster)
    # get pixel type/min/max/cell size info (r = arcpy.Raster() loads raster dataset into a Raster object so arcpy can access stats)
    r = arcpy.Raster(raster)
    # get spatial reference info
    sr = d.spatialReference
    vcs = sr.VCS.name if sr.VCS else "NONE"

    # info print block for raster
    print(f"\n---------------- {os.path.basename(raster)} ----------------: \n")
    print(f"   Number bands = {d.bandCount}")
    print(f"   Pixel type = {r.pixelType}")
    print(f"   ⬇️ Min = {r.minimum}")
    print(f"   ⬆️ Max = {r.maximum}")
    print(f"   📐 Cell size = {r.meanCellWidth}")
    print(f"   Horizontal SR: {sr.name}")
    print(f"   Vertical SR: {vcs}")
    print(f"\n")


# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1; skip rasters with existing stats calculated)
for stats_path1 in [gbc_bathy, gbc_slope]:
    arcpy.management.CalculateStatistics(
        in_raster_dataset=stats_path1,
        skip_existing="SKIP_EXISTING"
    )

# Call function to print info for each input GBC raster before removing overlap
print_raster_info(gbc_bathy)
print_raster_info(gbc_slope)


---------------- GBC15a_bathy_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8619.0
   ⬆️ Max = 0.0
   📐 Cell size = 472.8452741843477
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height



---------------- GBC15a_slope_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = 1.0411026707402016e-08
   ⬆️ Max = 79.22057342529297
   📐 Cell size = 472.845274184348
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height




In [51]:
# ── 3. CREATE GBC RASTERS WITH USC OVERLAP REMOVED ───────────────────────────────────────────────────

# Extract by Mask (Spatial Analyst Tools): Function
def extract_by_mask(in_raster, in_mask, out_raster):
    # snap raster to input raster grid so output aligns to same grid
    arcpy.env.snapRaster = in_raster
    # force output extent to full GBC raster so output isn't clipped to mask rectangle
    arcpy.env.extent = in_raster

    try:
        out_extract = ExtractByMask(
            in_raster=in_raster,
            in_mask_data=in_mask,
            # "OUTSIDE" writes only cells *outside of mask polygon* to output raster (all cells inside USC mask polygon become NoData in output raster)
            extraction_area="OUTSIDE"
        )
        # save output as new raster
        out_extract.save(out_raster)
        print(f"✅ No overlap GBC raster successfully created: {out_raster} \n")

    # print warning message if tool fails
    except Exception as e:
        print(f"❌[FAILURE] No-overlap raster creation failed for {os.path.basename(out_raster)} raster: {e} \n")


# Call function for GBC bathy + slope rasters
extract_by_mask(gbc_bathy, outline_usc_bathy, NOOVRLP_gbc_bathy)
extract_by_mask(gbc_slope, outline_usc_slope, NOOVRLP_gbc_slope)

# Clear settings in environment for subsequent blocks
arcpy.ClearEnvironment("snapRaster")
arcpy.ClearEnvironment("extent")

✅ No overlap GBC raster successfully created: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\nooverlap_GBC15a_bathy_p26920_v6642 

✅ No overlap GBC raster successfully created: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\nooverlap_GBC15a_slope_p26920_v6642 



In [52]:
# ── INSPECT GBC RASTERS DATA AFTER REMOVING USC OVERLAP ──────────────────────────────────────────────

# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1)
for stats_path2 in [NOOVRLP_gbc_bathy, NOOVRLP_gbc_slope]:
    arcpy.management.CalculateStatistics(in_raster_dataset=stats_path2)

# Call function to print info for each output GBC raster with overlap removed
print_raster_info(NOOVRLP_gbc_bathy)
print_raster_info(NOOVRLP_gbc_slope)


---------------- nooverlap_GBC15a_bathy_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8619.0
   ⬆️ Max = 0.0
   📐 Cell size = 472.8452741843477
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height



---------------- nooverlap_GBC15a_slope_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = 1.0411026707402016e-08
   ⬆️ Max = 73.05693817138672
   📐 Cell size = 472.845274184348
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height


